# fitness-rl — end-to-end walkthrough

This notebook reproduces the full pipeline in 6 cells: **data → world model → REINFORCE → A2C → PPO → 28-day recommendation**. Every cell is self-contained against the SDK and reads `configs/setup.json` for hyperparameters.

> Tip: open from the repo root with `jupyter notebook notebooks/fitness_rl_walkthrough.ipynb`. The SDK auto-resolves paths from the project root.

## 1 — Load the data pipeline

`DataService` walks the Kaggle CSVs → trajectory → 16-d state vectors. Reproducible: seed is fixed in `configs/setup.json`.

In [ ]:
import json, sys
from pathlib import Path

# Allow running the notebook from anywhere by walking up to the assignment root.
ROOT = next(p for p in Path.cwd().resolve().parents if (p / 'configs' / 'setup.json').exists()) \
       if not (Path.cwd() / 'configs' / 'setup.json').exists() else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from fitness_rl.sdk.sdk import FitnessRL
from fitness_rl.sdk.evaluator import FitnessRLEvaluator

sdk = FitnessRL(config_path=ROOT / 'configs' / 'setup.json')
data = sdk.prepare_data()
print(f'Chosen program: {data.chosen_title}')
print(f'Days: {len(data.trajectory)}  state dim: {data.states.shape[1]}')
print(f'Action distribution: ' + ', '.join(
    f"{a.name}={int((data.actions == int(a)).sum())}" for a in __import__('fitness_rl.shared.types', fromlist=['Action']).Action))

## 2 — Train the LSTM world model (Part C)

Supervised on rolling 7-day windows. Early stopping by validation loss.

In [ ]:
wm_result = sdk.train_world_model()
print(f'Best val MSE: {wm_result.best_val_loss:.6f} at epoch {wm_result.best_epoch}')
print(f'Early stopped: {wm_result.stopped_early}')

# Validate against persistence + linear baselines (audit #1)
ev = FitnessRLEvaluator(sdk)
wm_report = ev.evaluate_world_model(horizons=(1, 7))
print(f'Persistence 1-step MSE: {wm_report.persistence_one_step_mse:.4f}')
print(f'LSTM 1-step MSE:        {wm_report.lstm_one_step_mse:.4f}')
print(f'LSTM 7-step rollout MSE: {wm_report.lstm_rollout_mse[7]:.4f} (error compounds)')

## 3 — Train REINFORCE (Part D)

Episodic policy gradient with mean baseline + reward-to-go.

In [ ]:
reinforce_hist = sdk.train_reinforce(episodes=60)
print(f'REINFORCE — episodes: {len(reinforce_hist)}, final reward: {reinforce_hist[-1].total_reward:.3f}')

## 4 — Train A2C (Part E)

Per-step TD advantage with shared trunk + actor/critic heads.

In [ ]:
a2c_hist = sdk.train_a2c(episodes=60)
print(f'A2C — episodes: {len(a2c_hist)}, final reward: {a2c_hist[-1].total_reward:.3f}')

## 5 — Train PPO (Layer 15 — beyond-spec)

Clipped surrogate objective + multi-epoch updates per batch.

In [ ]:
ppo_hist = sdk.train_ppo(episodes=60)
print(f'PPO — episodes: {len(ppo_hist)}, final reward: {ppo_hist[-1].total_reward:.3f}')

# Side-by-side curve in one plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([m.total_reward for m in reinforce_hist], label='REINFORCE')
ax.plot([m.total_reward for m in a2c_hist], label='A2C')
ax.plot([m.total_reward for m in ppo_hist], label='PPO')
ax.set(xlabel='Episode', ylabel='Total reward',
       title='REINFORCE → A2C → PPO learning chain')
ax.legend(); plt.show()

## 6 — Recommend the next 7 days

The user-facing product: feed your recent workout history → get a 7-day plan with per-day reward breakdown.

In [ ]:
from fitness_rl.environment.reward import RewardFunction
from fitness_rl.services.recommender import WorkoutRecommender

cfg = sdk.config
reward_fn = RewardFunction(
    gain_weight=float(cfg.get('env.reward_gain_weight')),
    overload_lambda=float(cfg.get('env.reward_overload_lambda')),
    imbalance_lambda=float(cfg.get('env.reward_imbalance_lambda')),
)
plan = WorkoutRecommender().recommend(
    net=sdk._require_net('a2c'),
    env=sdk.make_env(),
    reward_fn=reward_fn,
    n_days=7,
    recent_actions=WorkoutRecommender.parse_history('PUSH,PUSH,REST'),
)
print(plan.as_table())

## What's next?

- Run `scripts/run_layer15_full.py` for the full 300-episode × 3-seed comparison.
- See `results/layer15/full_budget_multiseed.json` for the headline numbers and `assets/plots/three_algo_*.png` for the figures.
- See `README.md` § 9 for the full audit-driven empirical analysis.